# Week 5 - Q6: Where is margin concentrated?
**Question.** Engine Block EB-600 and Transmission Assembly TA-800 are about 36% of net revenue at below-average margin. Where is margin concentrated, what drags the portfolio margin, and what would a price change do?

**Approach.** Delivered orders only (18,033 orders, 130,402 lines of which 117,656 are delivered). Net revenue = `line_total` (ex-GST); gross profit = `line_profit` = quantity x (unit_price - unit_cost). Each of the 30 products is compared on its share of revenue and of gross profit, its margin, and its **contribution to the portfolio margin gap** (revenue share x (product margin - portfolio margin), which sums to exactly zero). Mix is then separated from rate over time, the 2024 fall in EB-600 is tested against what a scan of all 30 products would produce by chance, and a clearly labelled price what-if is run on the two products.

**Important property of the data.** Every line of a product carries the same unit price and the same unit cost for all six years (checked below). A product's margin is therefore fixed by the master price list: there is no sampling uncertainty in a product's margin, and the *only* way the portfolio margin can move over time is a change of mix. Pricing itself cannot be tested. Built on the corrected Week 2 merged datasets. Seed 42, 5,000 resamples.

In [1]:
import pandas as pd, numpy as np, os, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
from scipy import stats
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
SURF,INK,INK2,GRID,BLUE,ORANGE,AQUA,MUTED="#fcfcfb","#0b0b0b","#52514e","#e4e3df","#2a78d6","#eb6834","#1baf7a","#b9b8b2"
def find_dir(env_var, marker, rels):
    """Environment variable first, then this folder and every parent folder, using the repository layout."""
    if os.environ.get(env_var) and (Path(os.environ[env_var]) / marker).exists():
        return Path(os.environ[env_var])
    for base in [Path.cwd(), *Path.cwd().parents]:
        for rel in rels:
            if (base / rel / marker).exists():
                return base / rel
    return None
DATA_DIR = find_dir("CADETX_MERGED", "sales_orders_full.csv", ["merged", "data/merged", "week-02/reports/week_02/merged", "reports/week_02/merged"])
assert DATA_DIR is not None, "Could not find the merged datasets. Run from inside the repository, put a merged/ folder next to the notebook, or set the CADETX_MERGED environment variable to the folder."
OUT=Path("q6_outputs"); OUT.mkdir(exist_ok=True); CH=Path("q6_charts"); CH.mkdir(exist_ok=True)
SEED=42; NBOOT=5000; NPERM=5000
checks=[]
def check(name,expected,actual,ok):
    checks.append(dict(Check_Item=name,Expected=expected,Actual=actual,Status="PASS" if ok else "FAIL")); assert ok,f"{name}: expected {expected}, got {actual}"
def style(ax):
    ax.set_facecolor(SURF)
    for s in ["top","right"]: ax.spines[s].set_visible(False)
    for s in ["left","bottom"]: ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK2,labelsize=9)
print("Merged datasets:",DATA_DIR.resolve())

Merged datasets: <repository>/week-02/reports/week_02/merged


## Part 0. Load and integrity checks

In [2]:
lines=pd.read_csv(DATA_DIR/"sales_orders_full.csv",low_memory=False,parse_dates=["order_date","delivery_date"])
check("sales lines",130402,len(lines),len(lines)==130402)
check("unique orders",20000,lines.so_id.nunique(),lines.so_id.nunique()==20000)
D=lines[lines.order_status=="Delivered"].copy(); D["year"]=D.order_date.dt.year
check("delivered orders (Week 3)",18033,D.so_id.nunique(),D.so_id.nunique()==18033)
check("delivered lines",117656,len(D),len(D)==117656)
check("line_total = quantity x unit_price on every delivered line",0,int((D.line_total!=D.quantity*D.unit_price).sum()),(D.line_total!=D.quantity*D.unit_price).sum()==0)
check("line_profit = quantity x (unit_price - unit_cost) on every delivered line",0,int((D.line_profit!=D.quantity*(D.unit_price-D.unit_cost)).sum()),(D.line_profit!=D.quantity*(D.unit_price-D.unit_cost)).sum()==0)
REV=D.line_total.sum(); GP=D.line_profit.sum(); PM=GP/REV
check("net revenue is Rs 22.96B (Weeks 3 and 4)",22.96,round(REV/1e9,2),round(REV/1e9,2)==22.96)
check("gross profit is Rs 4.95B (Weeks 3 and 4)",4.95,round(GP/1e9,2),round(GP/1e9,2)==4.95)
check("portfolio margin is 21.56% (Weeks 3 and 4)",21.56,round(PM*100,2),round(PM*100,2)==21.56)
g=D.groupby("product_id")
check("30 products",30,D.product_id.nunique(),D.product_id.nunique()==30)
check("every product has exactly one unit price over six years",1,int(g.unit_price.nunique().max()),g.unit_price.nunique().max()==1)
check("every product has exactly one unit cost over six years",1,int(g.unit_cost.nunique().max()),g.unit_cost.nunique().max()==1)
print(f"Net revenue Rs {REV/1e9:.3f}B, gross profit Rs {GP/1e9:.3f}B, margin {PM*100:.2f}% on {D.so_id.nunique():,} delivered orders")

Net revenue Rs 22.959B, gross profit Rs 4.950B, margin 21.56% on 18,033 delivered orders


## Part 1. Product by product: revenue share, gross-profit share, margin and drag
Contribution to the portfolio gap = revenue share x (product margin - portfolio margin), in percentage points. A negative value pulls the 21.56% portfolio margin down; the 30 values sum to exactly zero.

In [3]:
P=D.groupby(["product_id","product_name","category"]).agg(units=("quantity","sum"),revenue=("line_total","sum"),gross_profit=("line_profit","sum"),unit_price=("unit_price","first"),unit_cost=("unit_cost","first")).reset_index()
P["margin_pct"]=P.gross_profit/P.revenue*100; P["revenue_share_pct"]=P.revenue/REV*100; P["gp_share_pct"]=P.gross_profit/GP*100
P["gp_minus_revenue_share_pts"]=P.gp_share_pct-P.revenue_share_pct
P["margin_vs_portfolio_pts"]=P.margin_pct-PM*100; P["contribution_to_gap_pts"]=P.revenue_share_pct/100*P.margin_vs_portfolio_pts
P["gp_if_portfolio_margin"]=P.revenue*PM; P["gp_shortfall_vs_portfolio_margin"]=P.gross_profit-P.gp_if_portfolio_margin
P=P.sort_values("revenue",ascending=False).reset_index(drop=True); P["revenue_rank"]=P.index+1; P["margin_rank"]=P.margin_pct.rank(ascending=False).astype(int)
P.to_csv(OUT/"q6_01_product_table.csv",index=False)
check("revenue shares add to 100%",100.0,round(P.revenue_share_pct.sum(),6),abs(P.revenue_share_pct.sum()-100)<1e-6)
check("gross-profit shares add to 100%",100.0,round(P.gp_share_pct.sum(),6),abs(P.gp_share_pct.sum()-100)<1e-6)
check("contributions to the margin gap add to zero",0.0,round(P.contribution_to_gap_pts.sum(),9),abs(P.contribution_to_gap_pts.sum())<1e-9)
check("product revenue adds to total",round(REV),round(P.revenue.sum()),abs(P.revenue.sum()-REV)<1)
eb=P[P.product_name.str.contains("EB-600")].iloc[0]; ta=P[P.product_name.str.contains("TA-800")].iloc[0]
check("EB-600 revenue Rs 4,856,985,000",4856985000,int(eb.revenue),int(eb.revenue)==4856985000)
check("EB-600 gross profit Rs 959,152,500",959152500,int(eb.gross_profit),int(eb.gross_profit)==959152500)
check("TA-800 revenue Rs 3,387,929,000",3387929000,int(ta.revenue),int(ta.revenue)==3387929000)
check("TA-800 gross profit Rs 608,610,000",608610000,int(ta.gross_profit),int(ta.gross_profit)==608610000)
two=P[P.product_id.isin([eb.product_id,ta.product_id])]; rest=P[~P.product_id.isin(two.product_id)]
t2r=two.revenue.sum()/REV*100; t2g=two.gross_profit.sum()/GP*100; rest_m=rest.gross_profit.sum()/rest.revenue.sum()*100
check("EB-600 and TA-800 are 35.9% of revenue (the sprint plan rounds this to 36.0%)",35.9,round(t2r,1),round(t2r,1)==35.9)
check("the other 28 products earn a 22.99% margin",22.99,round(rest_m,2),round(rest_m,2)==22.99)
check("EB-600 margin 19.75% and TA-800 margin 17.96%",(19.75,17.96),(round(eb.margin_pct,2),round(ta.margin_pct,2)),(round(eb.margin_pct,2),round(ta.margin_pct,2))==(19.75,17.96))
print(P[["revenue_rank","product_name","revenue_share_pct","gp_share_pct","margin_pct","contribution_to_gap_pts"]].round(2).head(10).to_string(index=False))
big=P.sort_values("contribution_to_gap_pts"); print("Biggest drags:",", ".join(f"{r.product_name} {r.contribution_to_gap_pts:+.2f}" for r in big.head(4).itertuples()),"| biggest lifts:",", ".join(f"{r.product_name} {r.contribution_to_gap_pts:+.2f}" for r in big.tail(3).itertuples()))
print(f"EB-600 + TA-800: {t2r:.2f}% of revenue, {t2g:.2f}% of gross profit; the other 28 products earn {rest_m:.2f}% against {PM*100:.2f}% overall")
cf=(two.revenue.sum()*PM-two.gross_profit.sum())
print(f"If both earned the portfolio margin, gross profit would be Rs {cf/1e6:.0f}M higher ({cf/GP*100:.1f}% of gross profit); the combined drag is {two.contribution_to_gap_pts.sum():.2f} points of the portfolio margin")

 revenue_rank                 product_name  revenue_share_pct  gp_share_pct  margin_pct  contribution_to_gap_pts
            1          Engine Block EB-600              21.16         19.38       19.75                    -0.38
            2 Transmission Assembly TA-800              14.76         12.30       17.96                    -0.53
            3         Boom Cylinder BC-400               8.42          8.96       22.94                     0.12
            4          Swing Motor SWM-350               7.20          7.22       21.61                     0.00
            5            Track Chain TC-45               7.09          6.03       18.34                    -0.23
            6          Turbocharger TB-900               6.43          5.13       17.19                    -0.28
            7    Hydraulic Cylinder HC-200               5.06          5.19       22.10                     0.03
            8        Hydraulic Pump HP-300               4.41          4.63       22.65         

## Part 2. Categories and concentration

In [4]:
Cg=D.groupby("category").agg(revenue=("line_total","sum"),gross_profit=("line_profit","sum"),products=("product_id","nunique")).reset_index()
Cg["margin_pct"]=Cg.gross_profit/Cg.revenue*100; Cg["revenue_share_pct"]=Cg.revenue/REV*100; Cg["gp_share_pct"]=Cg.gross_profit/GP*100; Cg["contribution_to_gap_pts"]=Cg.revenue_share_pct/100*(Cg.margin_pct-PM*100)
Cg=Cg.sort_values("revenue",ascending=False); Cg.to_csv(OUT/"q6_02_category_table.csv",index=False)
check("15 categories",15,len(Cg),len(Cg)==15)
check("category contributions add to zero",0.0,round(Cg.contribution_to_gap_pts.sum(),9),abs(Cg.contribution_to_gap_pts.sum())<1e-9)
eng=Cg[Cg.category=="Engine"].iloc[0]; hyd=Cg[Cg.category=="Hydraulic"].iloc[0]; tr=Cg[Cg.category=="Transmission"].iloc[0]
check("category revenue shares: Engine 33.04%, Hydraulic 28.40%, Transmission 14.76%",(33.04,28.40,14.76),(round(eng.revenue_share_pct,2),round(hyd.revenue_share_pct,2),round(tr.revenue_share_pct,2)),(round(eng.revenue_share_pct,2),round(hyd.revenue_share_pct,2),round(tr.revenue_share_pct,2))==(33.04,28.40,14.76))
s=np.sort(P.revenue.values)/REV; n=len(s); gini_r=1-2*np.sum(np.cumsum(s)-s/2)/n
sg=np.sort(P.gross_profit.values)/GP; gini_g=1-2*np.sum(np.cumsum(sg)-sg/2)/n
hhi_r=float(((P.revenue_share_pct/100)**2).sum()); hhi_g=float(((P.gp_share_pct/100)**2).sum())
rho,p_rho=stats.spearmanr(P.revenue,P.margin_pct); rho_p,p_rho_p=stats.spearmanr(P.unit_price,P.margin_pct)
CON=pd.DataFrame([dict(measure="Top 2 products, share of revenue (%)",value=P.revenue_share_pct.head(2).sum()),dict(measure="Top 2 products, share of gross profit (%)",value=P.sort_values("gross_profit",ascending=False).gp_share_pct.head(2).sum()),dict(measure="Top 5 products, share of revenue (%)",value=P.revenue_share_pct.head(5).sum()),dict(measure="Top 5 products, share of gross profit (%)",value=P.sort_values("gross_profit",ascending=False).gp_share_pct.head(5).sum()),dict(measure="Gini of revenue across 30 products",value=gini_r),dict(measure="Gini of gross profit across 30 products",value=gini_g),dict(measure="Herfindahl index of revenue (equal shares = 0.033)",value=hhi_r),dict(measure="Herfindahl index of gross profit",value=hhi_g),dict(measure="Spearman, product revenue vs margin",value=rho),dict(measure="  p-value",value=p_rho),dict(measure="Spearman, unit price vs margin",value=rho_p),dict(measure="  p-value",value=p_rho_p),dict(measure="Lowest product margin (%)",value=P.margin_pct.min()),dict(measure="Highest product margin (%)",value=P.margin_pct.max())])
CON.to_csv(OUT/"q6_03_concentration.csv",index=False)
check("gross profit is less concentrated in the top 2 than revenue is",True,(round(CON.value[0],1),round(CON.value[1],1)),CON.value[1]<CON.value[0])
print(Cg[["category","products","revenue_share_pct","gp_share_pct","margin_pct","contribution_to_gap_pts"]].round(2).to_string(index=False))
print(CON.round(3).to_string(index=False))

     category  products  revenue_share_pct  gp_share_pct  margin_pct  contribution_to_gap_pts
       Engine         4              33.04         30.84       20.13                    -0.47
    Hydraulic         6              28.40         29.82       22.64                     0.31
 Transmission         1              14.76         12.30       17.96                    -0.53
Undercarriage         4              13.14         13.95       22.89                     0.17
   Electrical         3               5.08          5.65       23.99                     0.12
      Cooling         1               1.86          2.03       23.53                     0.04
        Cabin         1               1.40          1.87       28.66                     0.10
      Sensors         1               0.73          1.04       30.38                     0.06
       Brakes         1               0.44          0.55       26.53                     0.02
      Filters         2               0.36          0.59    

## Part 3. Mix against rate over time
Because each product's margin is fixed, the portfolio margin can only change through mix. The yearly portfolio margin is shown, then tested: if year labels are shuffled across orders, how big a spread between the best and worst year appears by chance?

In [5]:
Y=D.groupby("year").agg(revenue=("line_total","sum"),gross_profit=("line_profit","sum")).reset_index(); Y["margin_pct"]=Y.gross_profit/Y.revenue*100
Y["revenue_change"]=Y.revenue.diff(); Y["gp_change"]=Y.gross_profit.diff()
base=D[D.year==2019].groupby("product_id").agg(r=("line_total","sum"),g=("line_profit","sum")); mrate=(base.g/base.r)
rows=[]
for yr,gp_ in D.groupby("year"):
    pr=gp_.groupby("product_id").agg(r=("line_total","sum"),g=("line_profit","sum")); share=pr.r/pr.r.sum(); rows.append(dict(year=yr,margin_at_fixed_2019_mix=float((pr.g.sum()/pr.r.sum())*100),eng_share_pct=float(gp_[gp_.category=="Engine"].line_total.sum()/pr.r.sum()*100),eb600_share_pct=float(pr.r[eb.product_id]/pr.r.sum()*100),ta800_share_pct=float(pr.r[ta.product_id]/pr.r.sum()*100)))
Y=Y.merge(pd.DataFrame(rows).drop(columns="margin_at_fixed_2019_mix"),on="year"); Y.to_csv(OUT/"q6_04_yearly_margin_and_mix.csv",index=False)
spread=Y.margin_pct.max()-Y.margin_pct.min()
check("yearly portfolio margin stays within 21.5% to 21.6% in every year",(21.5,21.6),(round(Y.margin_pct.min(),2),round(Y.margin_pct.max(),2)),Y.margin_pct.min()>=21.5 and Y.margin_pct.max()<=21.6)
O=D.groupby("so_id").agg(r=("line_total","sum"),g=("line_profit","sum"),year=("year","first")); rng=np.random.default_rng(SEED)
yrs=O.year.values; uy=np.unique(yrs); idx={y:np.where(yrs==y)[0] for y in uy}
def spread_of(lbl):
    m=[O.g.values[lbl==y].sum()/O.r.values[lbl==y].sum() for y in uy]; return (max(m)-min(m))*100
obs_sp=spread_of(yrs); perm=np.array([spread_of(rng.permutation(yrs)) for _ in range(NPERM)]); p_mix=float((perm>=obs_sp).mean())
check("the year-to-year spread of portfolio margin is no larger than shuffled years produce (permutation p > 0.05)",">0.05",round(p_mix,3),p_mix>0.05)
pd.DataFrame([dict(measure="Spread of yearly portfolio margin (points)",value=obs_sp),dict(measure="Typical spread with shuffled year labels (median, points)",value=float(np.median(perm))),dict(measure="Permutation p-value",value=p_mix)]).to_csv(OUT/"q6_05_mix_permutation.csv",index=False)
print(Y[["year","revenue","gross_profit","margin_pct","eng_share_pct","eb600_share_pct","ta800_share_pct"]].round(3).to_string(index=False))
print(f"Portfolio margin runs {Y.margin_pct.min():.3f}% to {Y.margin_pct.max():.3f}%, a spread of {obs_sp:.3f} points; shuffled years give {np.median(perm):.3f} typically (permutation p = {p_mix:.2f}). Mix shift is real by construction but tiny: the margin is stable.")

 year    revenue  gross_profit  margin_pct  eng_share_pct  eb600_share_pct  ta800_share_pct
 2019 3870016600     834440040      21.562         33.206           21.023           13.882
 2020 3917532120     845098840      21.572         33.441           21.227           14.897
 2021 3852479620     831184120      21.575         32.487           21.008           14.749
 2022 3816288220     821097090      21.516         33.170           21.547           15.349
 2023 3824675310     824558720      21.559         33.611           21.590           14.285
 2024 3677995560     793627930      21.578         32.297           20.513           15.411
Portfolio margin runs 21.516% to 21.578%, a spread of 0.062 points; shuffled years give 0.091 typically (permutation p = 0.84). Mix shift is real by construction but tiny: the margin is stable.


## Part 4. The 2024 fall in EB-600 (Q1) against a scan of all 30 products
EB-600 fell Rs 71.3M from 2023 to 2024. It was picked out after looking, so the fair question is whether the *largest* fall among 30 products would be this big by chance. Order labels (2023 or 2024) are shuffled 5,000 times and the largest absolute product change recorded each time.

In [6]:
Dd=D.copy(); mat=Dd.pivot_table(index="so_id",columns="product_id",values="line_total",aggfunc="sum",fill_value=0)
oy=D.groupby("so_id").year.first().reindex(mat.index); sel=oy.isin([2023,2024]).values; M2=mat.values[sel]; z24=(oy[sel]==2024).values
tot=M2.sum(0); delta=M2[z24].sum(0)-M2[~z24].sum(0); prods=mat.columns
chg=pd.DataFrame({"product_id":prods,"revenue_2023":M2[~z24].sum(0),"revenue_2024":M2[z24].sum(0),"change":delta}).merge(P[["product_id","product_name","category","margin_pct"]],on="product_id")
chg["gp_change"]=chg.change*chg.margin_pct/100; chg["share_of_total_change_pct"]=chg.change/chg.change.sum()*100; chg=chg.sort_values("change").reset_index(drop=True); chg.to_csv(OUT/"q6_06_change_2023_to_2024_by_product.csv",index=False)
ebc=float(chg.loc[chg.product_name.str.contains("EB-600"),"change"].iloc[0])
check("EB-600 revenue fell Rs 71.281M from 2023 to 2024 (Q1: Rs 71.3M)",-71281000,int(round(ebc)),int(round(ebc))==-71281000)
check("total revenue fell Rs 146.7M from 2023 to 2024",-146680000,int(round(chg.change.sum()/1000)*1000),abs(chg.change.sum()+146679750)<1000)
n24=int(z24.sum()); rngp=np.random.default_rng(SEED); mx=np.empty(NPERM)
for b in range(NPERM):
    zz=np.zeros(len(z24),bool); zz[rngp.choice(len(z24),n24,replace=False)]=True; d=2*M2[zz].sum(0)-tot; mx[b]=np.abs(d).max()
p_scan=float((mx>=np.abs(delta).max()).mean())
rb=np.random.default_rng(SEED); i23=np.where(~z24)[0]; i24=np.where(z24)[0]; col=list(prods).index(eb.product_id)
bs=np.array([M2[rb.choice(i24,len(i24)),col].sum()-M2[rb.choice(i23,len(i23)),col].sum() for _ in range(NBOOT)]); lo,hi=np.percentile(bs,[2.5,97.5])
tot_bs=np.array([M2[rb.choice(i24,len(i24))].sum()-M2[rb.choice(i23,len(i23))].sum() for _ in range(NBOOT)]); tlo,thi=np.percentile(tot_bs,[2.5,97.5])
check("the largest product change among 30 is no bigger than chance produces (scan permutation p > 0.05)",">0.05",round(p_scan,3),p_scan>0.05)
pd.DataFrame([dict(measure="EB-600 change 2023 to 2024 (Rs)",value=ebc),dict(measure="EB-600 95% bootstrap interval low",value=lo),dict(measure="EB-600 95% bootstrap interval high",value=hi),dict(measure="Total revenue change (Rs)",value=float(chg.change.sum())),dict(measure="Total change 95% interval low",value=tlo),dict(measure="Total change 95% interval high",value=thi),dict(measure="EB-600 share of the total fall (%)",value=ebc/chg.change.sum()*100),dict(measure="Largest absolute product change observed (Rs)",value=float(np.abs(delta).max())),dict(measure="Median of the largest change under shuffled labels (Rs)",value=float(np.median(mx))),dict(measure="Scan permutation p-value (30 products)",value=p_scan)]).to_csv(OUT/"q6_07_eb600_fall_tests.csv",index=False)
print(chg[["product_name","revenue_2023","revenue_2024","change","share_of_total_change_pct"]].head(5).round(0).to_string(index=False))
print(f"EB-600 change: Rs {ebc/1e6:+.1f}M (95% interval Rs {lo/1e6:+.1f}M to {hi/1e6:+.1f}M), {ebc/chg.change.sum()*100:.0f}% of the total change of Rs {chg.change.sum()/1e6:+.1f}M (interval Rs {tlo/1e6:+.1f}M to {thi/1e6:+.1f}M)")
print(f"Largest change among 30 products: Rs {np.abs(delta).max()/1e6:.1f}M observed, Rs {np.median(mx)/1e6:.1f}M typical under shuffling; scan permutation p = {p_scan:.2f}")

         product_name  revenue_2023  revenue_2024    change  share_of_total_change_pct
  Engine Block EB-600     825741000     754460000 -71281000                       49.0
  Swing Motor SWM-350     288908200     243735200 -45173000                       31.0
  Turbocharger TB-900     252815600     225209700 -27605900                       19.0
  Control Valve CV-75     121443400     112047000  -9396400                        6.0
Hydraulic Pump HP-300     171647000     162288000  -9359000                        6.0
EB-600 change: Rs -71.3M (95% interval Rs -174.1M to +29.9M), 49% of the total change of Rs -146.7M (interval Rs -308.1M to +10.0M)
Largest change among 30 products: Rs 71.3M observed, Rs 50.1M typical under shuffling; scan permutation p = 0.25


## Part 5. SCENARIO: what a price change on EB-600 and TA-800 would do
**This is a labelled what-if, not evidence.** Volume is held at the 2019 to 2024 level, cost per unit does not change, and the effect is on six years of Delivered gross profit. The data holds one price list and no price history, so there is no way to measure how customers would respond. The break-even volume loss (how much volume could fall before the price rise stops paying) is therefore shown beside every figure.

In [7]:
rows=[]
for r in two.itertuples():
    for pct in (1,2,3):
        p_=pct/100; new_gp=r.gross_profit+p_*r.revenue; be=p_*r.revenue/(r.gross_profit+p_*r.revenue)
        rows.append(dict(scenario_label="SCENARIO - price change with volume held; not evidence",product=r.product_name,price_change_pct=pct,base_revenue=r.revenue,base_gross_profit=r.gross_profit,gp_uplift=p_*r.revenue,new_gross_profit=new_gp,gp_uplift_pct=p_*r.revenue/r.gross_profit*100,new_margin_pct=new_gp/(r.revenue*(1+p_))*100,break_even_volume_loss_pct=be*100,uplift_as_pct_of_portfolio_gp=p_*r.revenue/GP*100))
W=pd.DataFrame(rows); W.to_csv(OUT/"q6_08_price_scenario.csv",index=False)
w1=W[(W["product"].str.contains("EB-600"))&(W.price_change_pct==1)].iloc[0]
check("SCENARIO: +1% on EB-600 adds exactly 1% of its revenue (Rs 48,569,850)",48569850,int(w1.gp_uplift),int(w1.gp_uplift)==48569850)
chk=W.iloc[0]; vol=(1-chk.break_even_volume_loss_pct/100); gp_be=vol*(chk.base_revenue/ (chk.base_revenue/chk.base_revenue))  # placeholder to keep arithmetic explicit below
r0=two[two.product_name.str.contains("EB-600")].iloc[0]; q_loss=chk.break_even_volume_loss_pct/100; gp_at_be=(1-q_loss)*(r0.revenue*1.01-(r0.revenue-r0.gross_profit))
check("SCENARIO: at the break-even volume loss, gross profit returns to its base value",round(r0.gross_profit),round(gp_at_be),abs(gp_at_be-r0.gross_profit)<1)
check("SCENARIO: every scenario row carries the scenario label",6,int(W.scenario_label.str.startswith("SCENARIO").sum()),W.scenario_label.str.startswith("SCENARIO").all())
print(W[["product","price_change_pct","gp_uplift","gp_uplift_pct","new_margin_pct","break_even_volume_loss_pct","uplift_as_pct_of_portfolio_gp"]].round(2).to_string(index=False))

                     product  price_change_pct   gp_uplift  gp_uplift_pct  new_margin_pct  break_even_volume_loss_pct  uplift_as_pct_of_portfolio_gp
         Engine Block EB-600                 1  48569850.0           5.06           20.54                        4.82                           0.98
         Engine Block EB-600                 2  97139700.0          10.13           21.32                        9.20                           1.96
         Engine Block EB-600                 3 145709550.0          15.19           22.09                       13.19                           2.94
Transmission Assembly TA-800                 1  33879290.0           5.57           18.78                        5.27                           0.68
Transmission Assembly TA-800                 2  67758580.0          11.13           19.57                       10.02                           1.37
Transmission Assembly TA-800                 3 101637870.0          16.70           20.35                 

## Part 6. The product master's `margin_percentage` is a markup, not a margin
The master carries a `margin_percentage` field. If it were a margin on price it would match the margin computed above; it does not.

In [8]:
pm=D.groupby("product_id").agg(master_pct=("margin_percentage","first"),price=("unit_price","first"),cost=("unit_cost","first"),name=("product_name","first")).reset_index()
pm["markup_pct"]=(pm.price/pm.cost-1)*100; pm["margin_on_price_pct"]=(1-pm.cost/pm.price)*100
pm["matches_markup"]=(pm.master_pct-pm.markup_pct).abs()<=0.11; pm["matches_margin"]=(pm.master_pct-pm.margin_on_price_pct).abs()<=0.11; pm.to_csv(OUT/"q6_09_master_margin_definition.csv",index=False)
nm=int(pm.matches_markup.sum()); ng=int(pm.matches_margin.sum())
odd=pm[~pm.matches_markup]
check("the master margin_percentage matches markup on cost (within 0.1 point) for 29 of 30 products",29,nm,nm==29)
check("the one exception is Engine Oil Filter OF-90 (master 37.3% vs markup 60.0%)","OF-90",list(odd.name),list(odd.name)==["Engine Oil Filter OF-90"])
check("the master margin_percentage matches margin on price (within 0.1 point) for none of the 30 products",0,ng,ng==0)
print(f"Master margin_percentage = markup on cost for {nm} of 30 products (within 0.1 point) and = margin on price for {ng}. Exception: {odd.name.iloc[0]} (master {odd.master_pct.iloc[0]:.1f}%, markup {odd.markup_pct.iloc[0]:.1f}%, margin on price {odd.margin_on_price_pct.iloc[0]:.1f}%). EB-600: master {pm.master_pct[pm.name.str.contains('EB-600')].iloc[0]:.1f}% vs margin on price {pm.margin_on_price_pct[pm.name.str.contains('EB-600')].iloc[0]:.2f}%. Always use gross profit / revenue, not the master field, for margin.")

Master margin_percentage = markup on cost for 29 of 30 products (within 0.1 point) and = margin on price for 0. Exception: Engine Oil Filter OF-90 (master 37.3%, markup 60.0%, margin on price 37.5%). EB-600: master 24.6% vs margin on price 19.75%. Always use gross profit / revenue, not the master field, for margin.


## Part 7. Charts

In [9]:
Q=P.sort_values("margin_pct")
fig,ax=plt.subplots(figsize=(8.4,7.4),facecolor=SURF); style(ax)
cols=[ORANGE if n in (eb.product_name,ta.product_name) else BLUE for n in Q.product_name]
ax.barh(Q.product_name,Q.margin_pct,color=cols,height=0.72); ax.axvline(PM*100,color=INK,lw=1.2); ax.text(PM*100+0.4,0.2,f"portfolio {PM*100:.2f}%",color=INK,fontsize=9)
ax.set_xlabel("Gross margin on price (%)",color=INK2); ax.grid(axis="x",color=GRID,lw=0.7); ax.set_axisbelow(True); ax.tick_params(axis="y",labelsize=8)
ax.set_title("Margin by product: EB-600 and TA-800 (orange) sit below the portfolio line",loc="left",color=INK,fontsize=12,fontweight="bold")
plt.tight_layout(); plt.savefig(CH/"q6_01_margin_by_product.png",dpi=150,facecolor=SURF); plt.close()
R=P.sort_values("contribution_to_gap_pts"); fig,ax=plt.subplots(figsize=(8.4,7.4),facecolor=SURF); style(ax)
ax.barh(R.product_name,R.contribution_to_gap_pts,color=[ORANGE if v<0 else AQUA for v in R.contribution_to_gap_pts],height=0.72); ax.axvline(0,color=INK,lw=1)
ax.set_xlabel("Contribution to portfolio margin gap (percentage points)",color=INK2); ax.grid(axis="x",color=GRID,lw=0.7); ax.set_axisbelow(True); ax.tick_params(axis="y",labelsize=8)
ax.set_title("What lifts and drags the 21.56% portfolio margin",loc="left",color=INK,fontsize=12,fontweight="bold")
plt.tight_layout(); plt.savefig(CH/"q6_02_contribution_to_margin_gap.png",dpi=150,facecolor=SURF); plt.close()
fig,ax=plt.subplots(figsize=(8.2,4.4),facecolor=SURF); style(ax); ax.grid(axis="y",color=GRID,lw=0.7); ax.set_axisbelow(True)
x=np.arange(3); w_=0.36
for off,(nm_,col) in zip((-w_/2,w_/2),[("Engine Block EB-600",BLUE),("Transmission Assembly TA-800",ORANGE)]):
    d=W[W["product"]==nm_]; ax.bar(x+off,d.gp_uplift/1e6,w_,color=col,label=nm_)
    for xi,v,be in zip(x+off,d.gp_uplift/1e6,d.break_even_volume_loss_pct): ax.text(xi,v+1,f"{v:.0f}M\nbreak-even\nloss {be:.1f}%",ha="center",fontsize=7.5,color=INK2)
ax.set_xticks(x); ax.set_xticklabels(["+1% price","+2% price","+3% price"]); ax.set_ylabel("Gross-profit uplift, Rs million",color=INK2); ax.set_ylim(0,190); ax.legend(frameon=False,fontsize=9,loc="upper left")
ax.set_title("SCENARIO only: volume held constant; not evidence",loc="left",color=INK,fontsize=12,fontweight="bold")
plt.tight_layout(); plt.savefig(CH/"q6_03_price_scenario.png",dpi=150,facecolor=SURF); plt.close()
print("charts saved:",sorted(p.name for p in CH.glob("*.png")))

charts saved: ['q6_01_margin_by_product.png', 'q6_02_contribution_to_margin_gap.png', 'q6_03_price_scenario.png']


## Part 8. Findings

In [10]:
print(f"- EB-600 and TA-800 are {t2r:.1f}% of net revenue and {t2g:.1f}% of gross profit. Their margins ({eb.margin_pct:.2f}% and {ta.margin_pct:.2f}%) are below the {PM*100:.2f}% portfolio margin and the other 28 products earn {rest_m:.2f}%. Together they pull the portfolio margin down by {-two.contribution_to_gap_pts.sum():.2f} points (TA-800 {ta.contribution_to_gap_pts:.2f}, EB-600 {eb.contribution_to_gap_pts:.2f}). Had both earned the portfolio margin, gross profit would be Rs {cf/1e6:.0f}M higher.")
print(f"- Margin by product ranges {P.margin_pct.min():.1f}% to {P.margin_pct.max():.1f}%. Revenue is concentrated (top 2 hold {t2r:.1f}%, against 6.7% if shares were equal) but gross profit is less so ({CON.value[1]:.1f}%): the largest products earn less per rupee. Across products, revenue and margin are negatively related (Spearman {rho:.2f}, p = {p_rho:.3f}).")
print(f"- Over time the margin is stable ({Y.margin_pct.min():.2f}% to {Y.margin_pct.max():.2f}%). Mix is the only thing that can move it and it moves it by {obs_sp:.2f} points (permutation p = {p_mix:.2f}).")
print(f"- The 2024 fall in EB-600 (Rs {-ebc/1e6:.1f}M, {ebc/chg.change.sum()*100:.0f}% of the total fall) is the largest of the 30 product changes but is not larger than a scan of 30 products would produce by chance (p = {p_scan:.2f}). It was picked after looking, so treat it as unconfirmed.")
print(f"- SCENARIO ONLY: a 1% price rise on EB-600 adds Rs {w1.gp_uplift/1e6:.1f}M of gross profit over six years with volume held (break-even volume loss {w1.break_even_volume_loss_pct:.1f}%). With no price history there is no way to say how volume would respond.")
print("- Data issue: the master margin_percentage is a markup on cost for 29 of 30 products (not a margin), and Engine Oil Filter OF-90 is inconsistent; do not use the field as a margin.")

- EB-600 and TA-800 are 35.9% of net revenue and 31.7% of gross profit. Their margins (19.75% and 17.96%) are below the 21.56% portfolio margin and the other 28 products earn 22.99%. Together they pull the portfolio margin down by 0.91 points (TA-800 -0.53, EB-600 -0.38). Had both earned the portfolio margin, gross profit would be Rs 210M higher.
- Margin by product ranges 17.2% to 45.8%. Revenue is concentrated (top 2 hold 35.9%, against 6.7% if shares were equal) but gross profit is less so (31.7%): the largest products earn less per rupee. Across products, revenue and margin are negatively related (Spearman -0.94, p = 0.000).
- Over time the margin is stable (21.52% to 21.58%). Mix is the only thing that can move it and it moves it by 0.06 points (permutation p = 0.84).
- The 2024 fall in EB-600 (Rs 71.3M, 49% of the total fall) is the largest of the 30 product changes but is not larger than a scan of 30 products would produce by chance (p = 0.25). It was picked after looking, so tr

## Part 9. Questions for the data owner
1. Is there a price history, list-price change or discount record? Every line carries the master price for all six years, so pricing cannot be tested.
2. Is `margin_percentage` in the product master meant to be a margin on price or a markup on cost? It equals the markup for 29 of 30 products; Engine Oil Filter OF-90 shows 37.3% where the markup is 60.0% (the margin on price is 37.5%), so that value looks wrong or on a different basis.
3. Is `unit_cost` a standard cost or the actual purchase cost? It is one value per product for six years, so cost changes cannot be seen.

## Part 10. Validation summary

In [11]:
V=pd.DataFrame(checks); V.to_csv("validation_report_week05_q6.csv",index=False)
print(f"{(V.Status=='PASS').sum()} of {len(V)} checks pass"); V

38 of 38 checks pass


,Check_Item,Expected,Actual,Status
0,sales lines,130402,130402,PASS
1,unique orders,20000,20000,PASS
2,delivered orders (Week 3),18033,18033,PASS
3,delivered lines,117656,117656,PASS
4,line_total = quantity x unit_price on every de...,0,0,PASS
5,line_profit = quantity x (unit_price - unit_co...,0,0,PASS
6,net revenue is Rs 22.96B (Weeks 3 and 4),22.96,22.96,PASS
7,gross profit is Rs 4.95B (Weeks 3 and 4),4.95,4.95,PASS
8,portfolio margin is 21.56% (Weeks 3 and 4),21.56,21.56,PASS
9,30 products,30,30,PASS
